<div align="center">

# Day 02: Simulation Foundations in Python

**Digital Twin with Python (VTR UGE 21), hands-on notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Day overview](https://github.com/utkukose/digital-twin-veltech-NB-lecture/tree/main/days/day-02) · [Lecture page](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html) · [Interactive lab](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lab.html) · [PDF notes](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/Day02_Lecture_Notes.pdf)

</div>

## How to use this notebook

This notebook is the hands-on part of the day. The concepts are explained on the [lecture page](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows; they have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from a Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures, in a few seconds to a few minutes. When the notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field, asdict
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 3.6), 'axes.grid': True, 'grid.alpha': 0.3})

def check(name, value, expected, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback."""
    if value is None:
        print(f"{name}: not answered yet. Replace None with your answer and run the cell again.")
        return False
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else "not yet. Compare your steps with the lecture and try again."))
    return ok



# The asset: a 48 V lithium-ion battery pack under a duty cycle.
#
#   states   SoC    state of charge, 0 to 1
#            T      cell temperature in degrees Celsius
#            Q      usable capacity in Ah, fades with use
#            R      internal resistance in ohm, grows as capacity fades
#   inputs   I(t)   current demanded by the load in A, positive when discharging
#            u      cooling-fan duty, 0 to 1
#            T_amb  ambient temperature in degrees Celsius
#
# The state of charge changes over minutes, the temperature over tens of minutes and
# the capacity over months.

@dataclass
class PackParams:
    Q0: float = 60.0        # Ah, capacity when new
    R0: float = 0.045       # Ohm, internal resistance when new
    m_cp: float = 9000.0    # J/K, thermal mass of the pack
    hA0: float = 2.2        # W/K, passive convection
    hA1: float = 12.0       # W/K, extra convection at full fan
    k_age: float = 5.5e-8   # Ah lost per A-second at the reference temperature
    Ea_R: float = 3800.0    # K, Arrhenius slope of the ageing law
    T_ref: float = 298.15   # K
    fan_W: float = 25.0     # W, electrical cost of the fan at full duty

    def as_vector(self):
        return np.array([self.R0, self.m_cp, self.hA0, self.hA1])


def pack_derivatives(t, x, I, u, T_amb, p: PackParams):
    """dx/dt for x = [SoC, T, Q, R]. Pure physics, no data, no noise."""
    SoC, T, Q, R = x
    T_K = T + 273.15
    P_gen = I ** 2 * R                                    # ohmic heating, W
    P_cool = (p.hA0 + p.hA1 * u) * (T - T_amb)            # convective loss, W
    arrh = np.exp(-p.Ea_R * (1.0 / T_K - 1.0 / p.T_ref))  # ageing accelerates with heat
    dSoC = -I / (3600.0 * max(Q, 1e-6))
    dT = (P_gen - P_cool) / p.m_cp
    dQ = -p.k_age * abs(I) * arrh
    dR = p.R0 * 0.9 * (p.Q0 - Q) / p.Q0 * 1e-3            # resistance tracks capacity fade
    return np.array([dSoC, dT, dQ, dR])


def duty_cycle(t, kind="urban", seed=RANDOM_STATE):
    """Current demand in amperes as a function of time in seconds."""
    rng = np.random.default_rng(seed)
    if kind == "urban":       # stop-start: bursts of traction, regenerative braking
        base = 14 + 26 * (np.sin(2 * np.pi * t / 420.0) > 0.15)
        regen = -18 * (np.sin(2 * np.pi * t / 420.0 + 0.9) > 0.93)
        return base + regen
    if kind == "highway":     # steady high load
        return 40 + 6 * np.sin(2 * np.pi * t / 1800.0)
    if kind == "idle":
        return 8 + 0 * t
    raise ValueError(kind)



def rk4_step(fun, t, x, dt, *args):
    k1 = fun(t, x, *args)
    k2 = fun(t + dt / 2, x + dt / 2 * k1, *args)
    k3 = fun(t + dt / 2, x + dt / 2 * k2, *args)
    k4 = fun(t + dt, x + dt * k3, *args)
    return x + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)


def simulate(p: PackParams, T_end=3600.0, dt=1.0, x0=None, cycle="urban",
             controller=None, T_amb=25.0, seed=RANDOM_STATE):
    """Roll the physical asset forward. `controller(t, x) -> fan duty` closes the loop."""
    x = np.array([0.95, T_amb + 1.0, p.Q0, p.R0]) if x0 is None else np.array(x0, float)
    n = int(T_end / dt)
    ts = np.arange(n + 1) * dt
    X = np.zeros((n + 1, 4)); X[0] = x
    U = np.zeros(n + 1); I_log = np.zeros(n + 1)
    for k in range(n):
        t = ts[k]
        I = float(duty_cycle(t, cycle, seed))
        u = 0.0 if controller is None else float(np.clip(controller(t, x), 0, 1))
        U[k] = u; I_log[k] = I
        x = rk4_step(pack_derivatives, t, x, dt, I, u, T_amb, p)
        x[0] = np.clip(x[0], 0.0, 1.0)
        X[k + 1] = x
    U[-1] = U[-2]; I_log[-1] = I_log[-2]
    return pd.DataFrame({"t": ts, "SoC": X[:, 0], "T": X[:, 1], "Q": X[:, 2],
                         "R": X[:, 3], "u": U, "I": I_log})



@dataclass
class SensorSpec:
    """Turns a true value into a reading: noise, bias, drift, quantisation, dropout and latency."""
    noise_sd: float = 0.0       # gaussian noise, in sensor units
    bias: float = 0.0           # constant offset
    drift_per_hour: float = 0.0 # slow calibration drift
    quantum: float = 0.0        # ADC quantisation step, 0 = none
    dropout: float = 0.0        # probability a sample is lost
    latency_s: float = 0.0      # transport delay


def read_sensor(t, true_values, spec: SensorSpec, rng):
    v = np.asarray(true_values, float).copy()
    v = v + spec.bias + spec.drift_per_hour * (t / 3600.0)
    if spec.noise_sd:
        v = v + rng.normal(0, spec.noise_sd, v.shape)
    if spec.quantum:
        v = np.round(v / spec.quantum) * spec.quantum
    if spec.dropout:
        v = np.where(rng.random(v.shape) < spec.dropout, np.nan, v)
    return v


print('pack model, integrator and sensors ready')

## Python step 2: NumPy arrays and functions that receive functions

NumPy stores numbers in arrays and computes with whole arrays at once &#91;[5](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html#ref-5)&#93;. This step builds a time grid for the duty cycle of the pack, the current that the load draws over time, and computes the heat that this current produces. It then writes a general Euler integrator that receives the model as a function and is checked against the exact solution.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "read_sensor" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-02/NB02_simulation_foundations.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

### Time grids and element-wise arithmetic

In [ ]:
import numpy as np
t = np.arange(0, 3600, 600)                  # every 10 minutes for one hour, in seconds
I = 40 + 6 * np.sin(2 * np.pi * t / 1800)    # current demanded at those times, in A
print(t)
print(np.round(I, 2))
print(I.shape, round(float(I.mean()), 2), round(float(I.max()), 2))

`np.arange(start, stop, step)` builds an array of evenly spaced values that stops before `stop`. Arithmetic and functions such as `np.sin` act on every element at once, so the current at all six times is computed in one line. `shape` gives the size of the array, and `mean` and `max` summarise it.

### Masks and summaries

In [ ]:
R = 0.045
heat = I ** 2 * R
print(np.round(heat, 1))
print("steps above 80 W:", int((heat > 80).sum()))
print("heat in the hottest step:", round(float(heat[heat.argmax()]), 1), "W at t =", int(t[heat.argmax()]), "s")

The line `heat = I ** 2 * R` is the ohmic heat $I^2 R$ of the lecture in watts, computed for all six currents at once. A comparison with an array returns an array of booleans, and its sum counts the `True` values. `argmax` returns the position of the largest element, and the same position indexes the matching time.

### A function that receives the model

In [ ]:
def euler(rate, x0, dt, n):
    """Explicit Euler for dx/dt = rate(t, x), n steps of size dt."""
    x, t_now = x0, 0.0
    for _ in range(n):
        x = x + dt * rate(t_now, x)
        t_now = t_now + dt
    return x

pack = lambda t_now, T: (40.0 ** 2 * 0.045 - 2.2 * (T - 35.0)) / 9000.0
print(f"Euler with 60 s steps: {euler(pack, 36.0, 60.0, 60):.4f} °C after one hour")

In Python a function is a value, so `euler` can receive the model `pack` as its argument `rate` and call it. A `lambda` writes a one-line function. The underscore in `for _ in range(n)` names a loop variable that is not used. The same integrator works for any model with the signature `rate(t, x)`. Here `pack` is the thermal equation of the lecture with a current of 40 A, the fan off and 35 °C outside, and the line `x = x + dt * rate(t_now, x)` is the Euler formula $x_{k+1} = x_k + \Delta t\, f(t_k, x_k)$.

### Checking against the exact solution

In [ ]:
T_inf = 35.0 + 40.0 ** 2 * 0.045 / 2.2      # equilibrium temperature
tau = 9000.0 / 2.2                           # time constant in seconds
exact = T_inf + (36.0 - T_inf) * np.exp(-3600 / tau)
print(f"exact: {exact:.4f} °C")
for dt in [600, 300, 150, 75]:
    err = abs(euler(pack, 36.0, dt, int(3600 / dt)) - exact)
    print(f"dt {dt:4d} s  error {err:.5f} °C")

With a constant current, the thermal model has an exact solution that decays exponentially towards the equilibrium temperature with the time constant $m c_p / hA_0$. In the code, `T_inf` is the equilibrium temperature $T_\infty$ of the lecture, `tau` the time constant $\tau$ and `exact` the exact solution after 3600 s. Halving the step halves the error, which is the signature of a first-order method.

### The order, measured

In [ ]:
e600 = abs(euler(pack, 36.0, 600, 6) - exact)
e300 = abs(euler(pack, 36.0, 300, 12) - exact)
print("observed order:", round(float(np.log2(e600 / e300)), 2))

The base-2 logarithm of the ratio of two errors, at step sizes that differ by a factor of two, estimates the order of the method. As a formula, $p \approx \log_2 (e(\Delta t) / e(\Delta t / 2))$, here with the errors at 600 s and 300 s. The notebook repeats this measurement for Heun and Runge-Kutta.

**Quick check.** Halving the step size divides the error by four. What is the order of the method?

<details><summary>Answer</summary>

2, because the error falls by 2 to the power of the order, and 2 squared is 4.

</details>

### Exercises for Python step 2

**Exercise 2.1.** Store the mean of the current array `I` in `I_mean`.

In [ ]:
I_mean = None   # your answer

check("Exercise 2.1", I_mean, 40.0)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
I_mean = float(I.mean())

check("Exercise 2.1", I_mean, 40.0)

**Exercise 2.2.** Store the largest value of `heat`, rounded to one decimal with `round`, in `heat_max`.

In [ ]:
heat_max = None   # your answer

check("Exercise 2.2", heat_max, 91.9)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
heat_max = round(float(heat.max()), 1)

check("Exercise 2.2", heat_max, 91.9)

**Exercise 2.3.** Measure the order of Euler from the step sizes 150 s and 75 s, round it to one decimal and store it in `order_small`.

In [ ]:
order_small = None   # your answer

check("Exercise 2.3", order_small, 1.0)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
e150 = abs(euler(pack, 36.0, 150, 24) - exact)
e75 = abs(euler(pack, 36.0, 75, 48) - exact)
order_small = round(float(np.log2(e150 / e75)), 1)

check("Exercise 2.3", order_small, 1.0)

## 1. Time scales of the pack

*Lecture: [Time scales](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html#time-scales)*

**From the formulas to the code.** `pack_derivatives`, defined in the setup cell, returns the four rates of the lecture in the order SoC, temperature, capacity and resistance. `x0` is the state at which they are evaluated, and `dx` holds the rates per second. The unit K/s is kelvin per second, the same as degrees Celsius per second. The line `tau = abs(x0[j] / v)` is the characteristic time $\tau_x = |x| / |dx/dt|$ in seconds, printed in minutes or in days. The test `if v != 0` avoids a division by zero: The resistance of a new pack does not change, so its time is infinite and is printed as `inf`. `np.log10` of the slowest time divided by the fastest is the spread in orders of magnitude.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "order_small" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-02/NB02_simulation_foundations.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
p = PackParams()
x0 = np.array([0.95, 36.0, p.Q0, p.R0])
dx = pack_derivatives(0.0, x0, I=40.0, u=0.0, T_amb=35.0, p=p)
names = ["dSoC/dt [1/s]", "dT/dt [K/s]", "dQ/dt [Ah/s]", "dR/dt [Ohm/s]"]
taus = []
for j, (n, v) in enumerate(zip(names, dx)):
    tau = abs(x0[j] / v) if v != 0 else np.inf
    taus.append(tau)
    unit = "min" if tau < 6e4 else "days"
    val = tau / 60 if tau < 6e4 else tau / 86400
    print(f"{n:16s} {v:+.6e}    characteristic time {val:10.1f} {unit}")
finite = [t for t in taus if np.isfinite(t) and t > 0]
print(f"\nfastest {min(finite)/60:.0f} min, slowest {max(finite)/86400:.0f} days "
      f"-> a spread of {np.log10(max(finite)/min(finite)):.1f} orders of magnitude.")
print("A spread of this size is called stiffness.")

**Python note.** `enumerate(zip(names, dx))` walks over pairs of a name and a rate and numbers them, so that `j` is the position, `n` the name and `v` the rate. `np.inf` is the floating-point value for infinity, which stands here for a quantity that does not change at all. `[t for t in taus if np.isfinite(t) and t > 0]` is a list comprehension with a condition: It collects the time constants that are finite and positive in a new list.

**Observe and reflect.** The capacity changes over months and the temperature within an hour. A twin that must predict the capacity after one year of trips: Which of the two quantities sets the length of its step, and which the length of its run? What does that mean for the number of steps?

## 2. Three integrators and their measured order

*Lecture: [Integration and its accuracy](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html#integration-and-its-accuracy)*

**Python move: functions as arguments.** Each step function receives the model as an argument, and `integrate` receives the step function.

**From the formulas to the code.** `euler_step`, `heun_step` and `rk4_step` are the three update formulas of the lecture for a general state `x`, and the slopes $s_1$ to $s_4$ are named `k1` to `k4`. RK4 is the classical Runge-Kutta method of order 4. `fun` is the right-hand side $f$, here `thermal_only`: the thermal equation alone, with 40 A, the fan off and 35 °C outside. `integrate` repeats one step `n` times from 36 °C, and its argument `T_end` is the simulated time in seconds, not a temperature. In the table `conv`, each order is `np.log2` of the error of the row above divided by the error of its own row, $p \approx \log_2 (e(\Delta t) / e(\Delta t / 2))$, so the first row has none. In the next cell, `3600 / dt_needed * work[nm]` is the number of derivative evaluations $N = n_s\, t_{\mathrm{end}} / \Delta t$ for the largest step of the table whose error is below 0.0001 °C.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "finite" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-02/NB02_simulation_foundations.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def euler_step(fun, t, x, dt, *a):
    return x + dt * fun(t, x, *a)

def heun_step(fun, t, x, dt, *a):
    k1 = fun(t, x, *a)
    k2 = fun(t + dt, x + dt * k1, *a)
    return x + dt / 2 * (k1 + k2)

def rk4_step(fun, t, x, dt, *args):
    k1 = fun(t, x, *args)
    k2 = fun(t + dt / 2, x + dt / 2 * k1, *args)
    k3 = fun(t + dt / 2, x + dt / 2 * k2, *args)
    k4 = fun(t + dt, x + dt * k3, *args)
    return x + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)


def simulate(p: PackParams, T_end=3600.0, dt=1.0, x0=None, cycle="urban",
             controller=None, T_amb=25.0, seed=RANDOM_STATE):
    """Roll the physical asset forward. `controller(t, x) -> fan duty` closes the loop."""
    x = np.array([0.95, T_amb + 1.0, p.Q0, p.R0]) if x0 is None else np.array(x0, float)
    n = int(T_end / dt)
    ts = np.arange(n + 1) * dt
    X = np.zeros((n + 1, 4)); X[0] = x
    U = np.zeros(n + 1); I_log = np.zeros(n + 1)
    for k in range(n):
        t = ts[k]
        I = float(duty_cycle(t, cycle, seed))
        u = 0.0 if controller is None else float(np.clip(controller(t, x), 0, 1))
        U[k] = u; I_log[k] = I
        x = rk4_step(pack_derivatives, t, x, dt, I, u, T_amb, p)
        x[0] = np.clip(x[0], 0.0, 1.0)
        X[k + 1] = x
    U[-1] = U[-2]; I_log[-1] = I_log[-2]
    return pd.DataFrame({"t": ts, "SoC": X[:, 0], "T": X[:, 1], "Q": X[:, 2],
                         "R": X[:, 3], "u": U, "I": I_log})


def thermal_only(t, x, I, u, T_amb, p):
    """Only the temperature state, so that clipping does not affect the convergence study."""
    T = x[0]
    return np.array([(I ** 2 * p.R0 - (p.hA0 + p.hA1 * u) * (T - T_amb)) / p.m_cp])

def integrate(step, dt, T_end=3600.0, I=40.0, u=0.0, T_amb=35.0, p=PackParams()):
    n = int(round(T_end / dt)); x = np.array([36.0])
    for k in range(n):
        x = step(thermal_only, k * dt, x, dt, I, u, T_amb, p)
    return float(x[0])

ref = integrate(rk4_step, 0.01)                # reference: RK4 at a very small step
print(f"reference T(3600 s) = {ref:.9f} C\n")

rows = []
for dt in (400.0, 200.0, 100.0, 50.0, 25.0):
    r = dict(dt=dt)
    for nm, st in (("euler", euler_step), ("heun", heun_step), ("rk4", rk4_step)):
        r[nm] = abs(integrate(st, dt) - ref)
    rows.append(r)
conv = pd.DataFrame(rows)
for nm in ("euler", "heun", "rk4"):
    conv[nm + "_order"] = np.r_[np.nan, np.log2(conv[nm].values[:-1] / conv[nm].values[1:])]
print(conv.to_string(index=False, float_format=lambda v: f"{v:.3e}" if abs(v) < 1 else f"{v:.2f}"))
print("\nmeasured orders (should approach 1, 2, 4):")
print("  euler", f"{conv.euler_order.iloc[-1]:.2f}   heun",
      f"{conv.heun_order.iloc[-1]:.2f}   rk4", f"{conv.rk4_order.iloc[-1]:.2f}")

**Python note.** `*a` in `def euler_step(fun, t, x, dt, *a)` collects all further arguments in a tuple, and `fun(t, x, *a)` passes them on unchanged. The three step functions therefore work for any model, whatever arguments it needs, and `integrate` can receive any of them as its argument `step`.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
for nm, col in (("euler", "#c0392b"), ("heun", "#f39c12"), ("rk4", "#2980b9")):
    ax.loglog(conv.dt, conv[nm], "o-", color=col, label=nm)
for slope, col in ((1, "#c0392b"), (2, "#f39c12"), (4, "#2980b9")):
    ref_line = conv[["euler", "heun", "rk4"][slope // 2]].iloc[0] * (conv.dt / conv.dt.iloc[0]) ** slope
    ax.loglog(conv.dt, ref_line, ls=":", lw=.8, color=col)
ax.set_xlabel("step size dt [s]"); ax.set_ylabel("absolute error at t = 3600 s")
ax.set_title("Order of accuracy, measured (dotted = theoretical slope)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

work = {"euler": 1, "heun": 2, "rk4": 4}
TOL = 1e-4
print(f"cost of reaching an error below {TOL} C over one hour:")
for nm in ("euler", "heun", "rk4"):
    dt_needed = next((d for d, e in zip(conv.dt, conv[nm]) if e < TOL), None)
    if dt_needed:
        print(f"   {nm:6s} dt <= {dt_needed:6.1f} s -> {3600 / dt_needed * work[nm]:8.0f} "
              f"derivative evaluations")
    else:
        print(f"   {nm:6s} needs a step smaller than {conv.dt.min():.1f} s "
              f"(> {3600 / conv.dt.min() * work[nm]:.0f} evaluations, tolerance not reached)")

**Observe and reflect.** With a step of 25 s, Runge-Kutta is within about $10^{-10}$ °C of the reference. The sensor of Day 1 has a noise of 0.35 °C. Which step and which method would you choose for the twin, and what does a smaller error than the sensor can ever show buy?

The next cell reads the measured order of Runge-Kutta from the last row of the table `conv`. `iloc[-1]` selects the last row, as the index `-1` selects the last element of a list.

In [ ]:
rk4_order = float(conv["rk4_order"].iloc[-1])
print(f"measured order of Runge-Kutta between 50 s and 25 s: {rk4_order:.2f}")

**Observe and reflect.** The orders measured between 400 s and 200 s lie further from 1, 2 and 4 than those of the last row. Why does the formula $e(\Delta t) \approx c\,\Delta t^{\,p}$ fit better for small steps?

## 3. The stability limit of explicit Euler

*Lecture: [The stability limit](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html#the-stability-limit)*

**From the formulas to the code.** In `stability_limit`, `lam` is the rate constant $\lambda = -(hA_0 + hA_1 u) / (m c_p)$, and the function returns the limit $2 / |\lambda|$ in seconds. The loop runs explicit Euler at full fan from 60 °C with steps of 0.5, 0.95, 1.05 and 1.3 times this limit. Every run takes `int(6000 / dt)` whole steps, so the four curves end at different times. The command `ax.set_ylim(0, 130)` cuts the plot at 0 °C, which hides the last point of the run at 1.3 times the limit.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "rk4_order" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-02/NB02_simulation_foundations.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def stability_limit(u, p=PackParams()):
    lam = -(p.hA0 + p.hA1 * u) / p.m_cp
    return 2.0 / abs(lam)

for u in (0.0, 0.25, 0.5, 1.0):
    print(f"fan duty {u:.2f} -> explicit-Euler stability limit dt <= {stability_limit(u):8.0f} s "
          f"({stability_limit(u)/60:.1f} min)")

print("\nexplicit Euler beyond the limit, fan at full duty:")
lim = stability_limit(1.0)
fig, ax = plt.subplots(figsize=(7.5, 3.4))
for dt, style in ((0.5 * lim, "-"), (0.95 * lim, "-"), (1.05 * lim, "--"), (1.3 * lim, ":")):
    n = int(6000 / dt); x = np.array([60.0]); traj = [60.0]
    for k in range(n):
        x = euler_step(thermal_only, k * dt, x, dt, 40.0, 1.0, 35.0, PackParams())
        traj.append(float(x[0]))
    ax.plot(np.arange(len(traj)) * dt / 60, traj, style,
            label=f"dt = {dt:.0f} s ({dt/lim:.2f} x limit)")
ax.set_ylim(0, 130); ax.set_xlabel("time [min]"); ax.set_ylabel("T [C]")
ax.set_title("Explicit Euler across its stability boundary"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Exercise A.** Use `stability_limit` to store the limit, in seconds, at a fan duty of 0.5 in `lim_half`.

In [ ]:
lim_half = None   # your answer

check("Exercise A", lim_half, float(stability_limit(0.5)))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
lim_half = float(stability_limit(0.5))

check("Exercise A", lim_half, float(stability_limit(0.5)))

**Observe and reflect.** The limit falls from 8182 s with the fan off to 1268 s at full fan. A twin that switches the fan on and off integrates with one fixed step. Which of the four printed limits must this step respect, and why is the limit with the fan off not enough?

## 4. Adaptive solvers &#91;[3](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html#ref-3), [4](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html#ref-4)&#93;

*Lecture: [Adaptive solvers](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html#adaptive-solvers)*

**From the formulas to the code.** `rhs`, short for right-hand side, is the function $f(t, x)$ of the pack model on the highway trip with the fan off. `solve_ivp` integrates it from 0 to 7200 s with the tolerances `rtol` and `atol` of the lecture. The three solvers are RK45, the explicit Runge-Kutta pair of orders 5 and 4, LSODA, the Livermore solver for ordinary differential equations with automatic method switching, and Radau, an implicit Runge-Kutta method. The printed `steps` is `len(sol.t)`, the number of time points with the start included, so RK45 takes 23 steps for its 24 points. `sol.nfev` counts the derivative evaluations. `solve_ivp` integrates the four equations as they stand and does not keep the state of charge between 0 and 1 as `simulate` does, so the cell compares the temperature `sol.y[1]` only. The times in milliseconds depend on the computer.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "lim_half" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-02/NB02_simulation_foundations.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
# An adaptive solver chooses the step itself. The methods LSODA and Radau are suited to stiff problems.
from scipy.integrate import solve_ivp
import time

def rhs(t, x, p, cycle, T_amb):
    return pack_derivatives(t, x, float(duty_cycle(t, cycle)), 0.0, T_amb, p)

p = PackParams(); x0 = [0.95, 36.0, p.Q0, p.R0]
for method in ("RK45", "LSODA", "Radau"):
    t0 = time.time()
    sol = solve_ivp(rhs, (0, 7200), x0, method=method, args=(p, "highway", 35.0),
                    rtol=1e-6, atol=1e-9, dense_output=True)
    el = time.time() - t0
    print(f"{method:6s} steps {len(sol.t):5d}   rhs evals {sol.nfev:6d}   "
          f"{el*1000:6.1f} ms   T(end) {sol.y[1,-1]:.4f} C")

fixed = simulate(p, T_end=7200, dt=1.0, cycle="highway", T_amb=35.0)
print(f"\nfixed-step RK4 at dt=1 s: 7200 steps, 28800 rhs evals, "
      f"T(end) {fixed['T'].iloc[-1]:.4f} C")
print("\nAdaptive solvers take large steps where the solution changes slowly and small steps where it changes fast.")
print("For a twin that runs in real time, a fixed step is often preferred,")
print("because a solver that sometimes takes 40 ms and sometimes 400 ms cannot meet a deadline.")

**Observe and reflect.** RK45 needs 23 steps for the trip, LSODA 102 and Radau 96. Which numbers of the printed lines matter for a twin on a small processor that must answer every second, and which for an offline study of a year of ageing?

## 5. A charging depot as a discrete-event simulation &#91;[1](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html#ref-1)&#93;

*Lecture: [A depot and a fleet](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html#a-depot-and-a-fleet)*

**Python move: a priority queue.** `heapq` keeps the future events ordered by time.

**From the formulas to the code.** The class `DES`, short for discrete-event simulation, is the event list of the lecture. Its method `schedule` puts an event with its time on the list `q`, and `run` takes the earliest one, sets the clock `t` and calls its action. The counter `seq` keeps two events with the same time in the order in which they were scheduled. `arrive` and `finish` are the two kinds of events, and `start` computes the charging time $t_c = (0.9 - \mathrm{SoC}_{\mathrm{in}})\, Q_0 / I_c$, converted to seconds. `wait_h` is the waiting time $w$ in hours, `p95_wait_h` the wait that 95 percent of the vehicles stay below, and `utilisation` is $\rho$, computed as the sum of all charging hours divided by `bays * 24`. The arrival times are drawn evenly over `T_end * 0.75`, the first 18 hours, and the run continues to hour 36 so that every vehicle is finished.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "fixed" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-02/NB02_simulation_foundations.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
import heapq
from collections import deque

class DES:
    """Minimal discrete-event simulator: a clock, a priority queue, and callbacks."""
    def __init__(self):
        self.t = 0.0; self.q = []; self.seq = 0; self.log = []
    def schedule(self, delay, fn, *a):
        self.seq += 1
        heapq.heappush(self.q, (self.t + delay, self.seq, fn, a))
    def run(self, until):
        while self.q and self.q[0][0] <= until:
            self.t, _, fn, a = heapq.heappop(self.q)
            fn(*a)
        self.t = until

def charging_depot(n_bays=3, n_vehicles=60, T_end=24 * 3600, seed=RANDOM_STATE):
    """A fixed demand of n_vehicles arrivals per day. The demand does not depend on the
    number of bays, so that depots with different numbers of bays can be compared."""
    rng = np.random.default_rng(seed)
    sim = DES(); free = n_bays; waiting = deque(); stats = []

    def arrive(v):
        nonlocal free
        soc = float(rng.uniform(0.05, 0.4))
        job = dict(v=v, soc=soc, t_arrive=sim.t)
        if free > 0:
            free -= 1; start(job)
        else:
            waiting.append(job)

    def start(job):
        job["t_start"] = sim.t
        charge_time = (0.9 - job["soc"]) * 60.0 * 3600 / 45.0      # Ah needed / 45 A charger
        sim.schedule(charge_time, finish, job)

    def finish(job):
        nonlocal free
        job["t_done"] = sim.t
        stats.append(dict(wait_h=(job["t_start"] - job["t_arrive"]) / 3600,
                          charge_h=(job["t_done"] - job["t_start"]) / 3600,
                          soc_in=job["soc"]))
        if waiting:
            start(waiting.popleft())
        else:
            free += 1

    # Poisson arrivals, concentrated in the working day
    times = np.sort(rng.uniform(0, T_end * 0.75, n_vehicles))
    for v, t in enumerate(times):
        sim.schedule(float(t), arrive, v)
    sim.run(T_end * 1.5)
    return pd.DataFrame(stats)

print("demand fixed at 60 vehicles/day; only the number of bays changes\n")
rows = []
for bays in (2, 3, 4, 5, 6, 8):
    st = charging_depot(n_bays=bays)
    rows.append(dict(bays=bays, completed=len(st), mean_wait_h=st.wait_h.mean(),
                     p95_wait_h=st.wait_h.quantile(.95),
                     utilisation=st.charge_h.sum() / (bays * 24)))
depot = pd.DataFrame(rows)
print(depot.round(3).to_string(index=False))

**Python note.** `nonlocal free` lets the inner functions `arrive` and `finish` change the variable `free` of `charging_depot`, which they could otherwise only read. A `deque` is a list that is quick to change at both ends: `append` adds a vehicle at the end of the line, and `popleft` takes the first one.

**Exercise B.** What is the smallest number of bays for which the mean wait in the table `depot` is below one hour? Store it in `bays_needed`.

In [ ]:
bays_needed = None   # your answer

check("Exercise B", bays_needed, int(depot[depot["mean_wait_h"] < 1]["bays"].min()))

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
bays_needed = int(depot[depot["mean_wait_h"] < 1]["bays"].min())

check("Exercise B", bays_needed, int(depot[depot["mean_wait_h"] < 1]["bays"].min()))

**Observe and reflect.** Each additional bay divides the mean wait by about three to four, yet the hours that it saves shrink quickly: 0.58 hours from three to four bays and 0.05 hours from five to six. How would you weigh the cost of a bay against the waiting time of the drivers?

## 6. A fleet as an agent-based simulation &#91;[2](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html#ref-2)&#93;

*Lecture: [A depot and a fleet](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html#a-depot-and-a-fleet)*

**From the formulas to the code.** Each `Vehicle` is one agent. Its array `x` is the state of its pack, `cycle` its trip pattern and `threshold` the state of charge below which it looks for a bay. The method `step` applies the three rules of the lecture. The line `self.x[0] + 45.0 * dt / (3600 * self.x[2])` is $\mathrm{SoC}_k + I_c\, \Delta t / (3600\, Q)$, the line after it cools the parked pack with one explicit Euler step of the thermal equation without heat input, and `np.clip((self.x[1] - 42) / 6, 0, 1)` is the fan rule. `step` returns $-1$ when the vehicle takes a bay and $+1$ when it leaves one, and `fleet_day` adds this number to `free`. A stranded vehicle is not removed: Its state of charge is held at 0 until it gets a bay. `hist.stranded.max()` is the largest number of vehicles that are stranded at the same time, and the last line prints the spread of the capacities in milliampere-hours (mAh), thousandths of an ampere-hour.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "bays_needed" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-02/NB02_simulation_foundations.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
class Vehicle:
    def __init__(self, vid, p, rng):
        self.id = vid; self.p = p
        self.x = np.array([rng.uniform(0.6, 1.0), 35 + rng.normal(0, 1.5), p.Q0, p.R0])
        self.cycle = rng.choice(["urban", "highway"], p=[0.7, 0.3])
        self.charging = False; self.km = 0.0
        self.threshold = float(rng.uniform(0.15, 0.30))   # heterogeneity: different drivers
    def step(self, t, dt, T_amb, bays_free):
        if self.charging:
            self.x[0] = min(1.0, self.x[0] + 45.0 * dt / (3600 * self.x[2]))
            self.x[1] += (0.0 - (self.p.hA0) * (self.x[1] - T_amb)) / self.p.m_cp * dt
            if self.x[0] >= 0.9:
                self.charging = False; return +1
            return 0
        if self.x[0] < self.threshold and bays_free > 0:
            self.charging = True; return -1
        I = float(duty_cycle(t, self.cycle))
        u = float(np.clip((self.x[1] - 42) / 6, 0, 1))
        self.x = rk4_step(pack_derivatives, t, self.x, dt, I, u, T_amb, self.p)
        self.x[0] = np.clip(self.x[0], 0, 1); self.km += 0.011 * dt
        return 0

def fleet_day(n=30, bays=4, dt=60.0, T_end=24 * 3600, T_amb=35.0, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    fleet = [Vehicle(i, PackParams(), rng) for i in range(n)]
    free = bays; hist = []
    for k in range(int(T_end / dt)):
        t = k * dt
        for v in fleet:
            free += v.step(t, dt, T_amb, free)
        hist.append(dict(t=t, charging=sum(v.charging for v in fleet), bays_free=free,
                         mean_soc=np.mean([v.x[0] for v in fleet]),
                         mean_T=np.mean([v.x[1] for v in fleet]),
                         stranded=sum(v.x[0] <= 0.01 for v in fleet)))
    return pd.DataFrame(hist), fleet

hist, fleet = fleet_day()
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(hist.t / 3600, hist.mean_soc, color="#2980b9", label="mean SoC")
ax[0].plot(hist.t / 3600, hist.stranded / 30, color="#c0392b", label="fraction stranded")
ax[0].set_xlabel("hour"); ax[0].legend(fontsize=8); ax[0].set_title("Fleet state")
ax[1].plot(hist.t / 3600, hist.charging, color="#27ae60")
ax[1].axhline(4, ls="--", lw=.8, color="grey")
ax[1].set_xlabel("hour"); ax[1].set_ylabel("vehicles charging")
ax[1].set_title("Bay occupancy (dashed = capacity)")
plt.tight_layout(); plt.show()

print(f"stranded vehicles at any point: {int(hist.stranded.max())} of 30")
print(f"peak simultaneous charging   : {int(hist.charging.max())} of 4 bays")
print(f"capacity spread after one day: "
      f"{np.std([v.x[2] for v in fleet]) * 1000:.2f} mAh across the fleet")

**Python note.** `rng.choice(["urban", "highway"], p=[0.7, 0.3])` draws one of the two words with the given probabilities. `step` returns +1 when a vehicle frees a bay, -1 when it takes one and 0 otherwise, so that `free += v.step(...)` keeps the count of the free bays up to date.

**Observe and reflect.** No rule of a single vehicle mentions the stranded vehicles. Change `fleet_day()` to `fleet_day(bays=8)` and run the cell again. Does the number of stranded vehicles fall in proportion to the bays, and what does the balance of charge of the lecture predict?

## 7. Your asset: accuracy and stability

*Lecture: [Your asset](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html#your-asset)*

Change `ASSET` to `server_rack`, `motor` or `greenhouse` and run the cell again.

**From the formulas to the code.** `f_asset` is the thermal equation of the lecture for the chosen asset at full cooling, with `asset["C"]` as the heat capacity $C$ and `asset["power"]` as the heat input $P(t)$. `limit_dt` is the stability limit $2C / (hA_0 + hA_1)$, and `integrate_asset` holds the Euler and the Runge-Kutta formula once more, written for a single number. For each share of the limit, the step is changed to one that divides the simulated time into `n` whole steps, which is why the table shows shares such as 0.947 and 1.136. The errors are the largest distance from the reference `T_asset_ref` over the whole run. For the server rack and the greenhouse the heat input has a corner where its sine wave is cut off at zero, and there the error of Runge-Kutta falls more slowly than its order predicts. `run_asset` belongs to the switch that all days share and is not used in this section.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "fleet" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-02/NB02_simulation_foundations.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-7":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
ASSET = "battery"        # choose: battery, server_rack, motor or greenhouse

ASSETS = {
    "battery":     dict(name="battery pack", C=9000.0, hA0=2.2, hA1=12.0, T_amb=35.0, T0=36.0, limit=46.0,
                        horizon=7200.0, dt=5.0, power=lambda t: (40 + 6 * np.sin(2 * np.pi * t / 1800)) ** 2 * 0.045),
    "server_rack": dict(name="server rack", C=1.2e5, hA0=40.0, hA1=400.0, T_amb=24.0, T0=26.0, limit=35.0,
                        horizon=7200.0, dt=10.0, power=lambda t: 3000 + 1500 * np.maximum(0, np.sin(2 * np.pi * t / 3600))),
    "motor":       dict(name="motor winding", C=4000.0, hA0=1.5, hA1=8.0, T_amb=30.0, T0=32.0, limit=90.0,
                        horizon=5400.0, dt=5.0, power=lambda t: (20 + 8 * np.sin(2 * np.pi * t / 1200)) ** 2 * 0.3),
    "greenhouse":  dict(name="greenhouse air", C=3.0e6, hA0=500.0, hA1=4000.0, T_amb=18.0, T0=20.0, limit=32.0,
                        horizon=86400.0, dt=60.0, power=lambda t: 30000 * np.maximum(0, np.sin(2 * np.pi * t / 86400))),
}

def run_asset(a, controller=None, params=None, sensor_sd=0.0, seed=0, T_end=None, dt=None):
    """Explicit Euler for dT/dt = (P(t) - (hA0 + hA1 u)(T - T_amb)) / C; the controller maps a reading to a duty u."""
    p = dict(a); p.update(params or {}); dt = dt or a["dt"]; n = int((T_end or a["horizon"]) / dt)
    t = np.arange(n) * dt; T = np.empty(n); u = np.zeros(n); rng = np.random.default_rng(seed); T[0] = a["T0"]
    for k in range(n - 1):
        reading = T[k] + sensor_sd * rng.normal()
        u[k] = controller(reading) if controller else 0.0
        T[k + 1] = T[k] + dt * (a["power"](t[k]) - (p["hA0"] + p["hA1"] * u[k]) * (T[k] - p["T_amb"])) / p["C"]
    return t, T, u


asset = ASSETS[ASSET]
f_asset = lambda t_, T_: (asset["power"](t_) - (asset["hA0"] + asset["hA1"]) * (T_ - asset["T_amb"])) / asset["C"]     # the asset at full cooling

def integrate_asset(method, n):
    """n steps of explicit Euler or classical Runge-Kutta over the whole horizon."""
    dt = asset["horizon"] / n; t = np.linspace(0, asset["horizon"], n + 1); T = np.empty(n + 1); T[0] = asset["T0"]
    for k in range(n):
        if method == "euler":
            T[k + 1] = T[k] + dt * f_asset(t[k], T[k])
        else:
            k1 = f_asset(t[k], T[k]); k2 = f_asset(t[k] + dt / 2, T[k] + dt / 2 * k1)
            k3 = f_asset(t[k] + dt / 2, T[k] + dt / 2 * k2); k4 = f_asset(t[k] + dt, T[k] + dt * k3)
            T[k + 1] = T[k] + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return t, T

t_asset_ref, T_asset_ref = integrate_asset("rk4", 20000)                                # a fine reference solution
limit_dt = 2 * asset["C"] / (asset["hA0"] + asset["hA1"])                         # stability limit of explicit Euler
asset_rows = []
for frac in (0.02, 0.05, 0.1, 0.25, 0.5, 0.9, 1.1):
    n = max(1, int(asset["horizon"] // (frac * limit_dt))); t_, T_e = integrate_asset("euler", n); _, T_r = integrate_asset("rk4", n)
    ref_asset = np.interp(t_, t_asset_ref, T_asset_ref)
    asset_rows.append(dict(step_s=asset["horizon"] / n, share_of_limit=asset["horizon"] / n / limit_dt,
                     euler_error=float(np.abs(T_e - ref_asset).max()), rk4_error=float(np.abs(T_r - ref_asset).max())))
asset_steps = pd.DataFrame(asset_rows)
print(f"{asset['name']}: stability limit of explicit Euler at full cooling {limit_dt:.0f} s")
print(asset_steps.round(5).to_string(index=False))
plt.figure(figsize=(6.5, 3.4))
plt.loglog(asset_steps.step_s, asset_steps.euler_error + 1e-12, "o-", label="explicit Euler")
plt.loglog(asset_steps.step_s, asset_steps.rk4_error + 1e-12, "s-", label="classical Runge-Kutta")
plt.axvline(limit_dt, color="#c0392b", ls="--", lw=1, label="Euler stability limit")
plt.xlabel("step size (s)"); plt.ylabel("largest error over the horizon (C)"); plt.legend(fontsize=8); plt.show()

The next cell finds the largest step of the table for which Euler stays within 1 °C of the reference. `asset_steps[asset_steps.euler_error < 1]` keeps the rows in which the condition holds, and `.step_s.max()` takes the largest step among them.

In [ ]:
safe_step = float(asset_steps[asset_steps.euler_error < 1].step_s.max())
print(f"{asset['name']}: largest step with an Euler error below 1 C: {safe_step:.0f} s, "
      f"{safe_step / limit_dt:.2f} of the stability limit")

**Observe and reflect.** Switch between the four assets and compare the safe step with the stability limit of each. For which asset does the safe step lie closest to the limit, and how does the length of the run change the answer?

## 8. Going further (optional): sensors as models and tests that encode physics &#91;[6](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html#ref-6)&#93;

*Lecture: [Going further (optional)](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lecture.html#going-further-optional)*

**From the formulas to the code.** `read_sensor`, defined in the setup cell, applies the formula of the lecture in four moves: bias and drift, noise, rounding to the resolution `quantum`, and dropout. A lost reading becomes `nan`, short for not a number, and the field `latency_s` is not applied. In the name of one sensor, ADC stands for analogue-to-digital converter. `bias_C` and `sd_C` are the mean and the standard deviation of the error, computed with `np.nanmean` and `np.nanstd`, which skip the lost readings. In the second cell, `TwinConfig` is the frozen configuration, and `fingerprint` returns the first 12 characters of its hash, computed with the Secure Hash Algorithm with 256 bits (SHA-256). `T_eq` in `test_thermal_equilibrium` is the equilibrium temperature $T_\infty$, and each `assert` stops the run with an error when its condition is false.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "safe_step" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-02/NB02_simulation_foundations.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-8":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
truth = simulate(PackParams(), T_end=3600, dt=1.0, cycle="highway", T_amb=35.0)
rng = np.random.default_rng(RANDOM_STATE)

specs = {
    "ideal":         SensorSpec(),
    "noisy":         SensorSpec(noise_sd=0.6),
    "biased":        SensorSpec(bias=-2.5),
    "drifting":      SensorSpec(drift_per_hour=2.0),
    "coarse ADC":    SensorSpec(quantum=1.0),
    "lossy link":    SensorSpec(noise_sd=0.3, dropout=0.25),
    "realistic":     SensorSpec(noise_sd=0.4, bias=-0.6, drift_per_hour=0.8,
                                quantum=0.5, dropout=0.05),
}
fig, axes = plt.subplots(2, 4, figsize=(13, 5), sharex=True, sharey=True)
rows = []
for ax, (nm, sp) in zip(axes.ravel(), specs.items()):
    rng = np.random.default_rng(RANDOM_STATE)
    meas = np.array([read_sensor(t, [T], sp, rng)[0]
                     for t, T in zip(truth.t.values, truth["T"].values)])
    ax.plot(truth.t / 60, truth["T"], lw=1.6, color="#111111")
    ax.plot(truth.t / 60, meas, lw=.7, color="#c0392b")
    ax.set_title(nm, fontsize=9)
    err = meas - truth["T"].values
    rows.append(dict(sensor=nm, bias_C=np.nanmean(err), sd_C=np.nanstd(err),
                     missing_pct=100 * np.mean(~np.isfinite(meas))))
axes.ravel()[-1].axis("off")
fig.suptitle("Truth (black) versus what the twin receives (red)", fontsize=10)
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Observe and reflect.** The drifting sensor is wrong by 1.0 °C on average after one hour and would be wrong by 2.0 °C after two. Which of the seven sensors could a twin notice by itself, by comparing the readings with its own prediction, and which could it never notice?

In [ ]:
# --- 1. Configuration as data -----------------------------------------------
@dataclass(frozen=True)
class TwinConfig:
    dt: float = 1.0
    horizon_s: float = 7200.0
    cycle: str = "highway"
    T_amb: float = 35.0
    sync_period_s: float = 60.0
    fan_on_C: float = 42.0
    seed: int = RANDOM_STATE
    def fingerprint(self):
        import hashlib, json
        return hashlib.sha256(json.dumps(asdict(self), sort_keys=True).encode()).hexdigest()[:12]

cfg = TwinConfig()
print("config fingerprint:", cfg.fingerprint(), "\n", asdict(cfg), "\n")

# --- 2. Tests that encode the physics --------------------------------------
def test_charge_conservation():
    """With no current, SoC must not move."""
    d = pack_derivatives(0, [0.5, 40, 60, 0.045], I=0.0, u=0.0, T_amb=35, p=PackParams())
    assert abs(d[0]) < 1e-15, d[0]

def test_thermal_equilibrium():
    """At the equilibrium temperature the derivative must vanish."""
    p = PackParams(); I, u, T_amb = 40.0, 0.0, 35.0
    T_eq = T_amb + I ** 2 * p.R0 / (p.hA0 + p.hA1 * u)
    d = pack_derivatives(0, [0.5, T_eq, p.Q0, p.R0], I, u, T_amb, p)
    assert abs(d[1]) < 1e-12, d[1]

def test_second_law():
    """Cooling can never heat the pack: More fan must never raise dT/dt."""
    p = PackParams()
    d0 = pack_derivatives(0, [0.5, 55, 60, .045], 40, 0.0, 35, p)[1]
    d1 = pack_derivatives(0, [0.5, 55, 60, .045], 40, 1.0, 35, p)[1]
    assert d1 <= d0, (d0, d1)

def test_ageing_is_monotone():
    """Capacity may only decrease, and faster when hot."""
    p = PackParams()
    cold = pack_derivatives(0, [0.5, 20, 60, .045], 40, 0, 20, p)[2]
    hot = pack_derivatives(0, [0.5, 60, 60, .045], 40, 0, 20, p)[2]
    assert cold <= 0 and hot <= 0 and hot < cold, (cold, hot)

def test_determinism():
    """The same configuration and seed must give a byte-identical trajectory."""
    a = simulate(PackParams(), T_end=600, dt=1.0, cycle="urban")
    b = simulate(PackParams(), T_end=600, dt=1.0, cycle="urban")
    assert np.allclose(a.values, b.values, atol=0, rtol=0)

def test_integrator_order():
    """RK4 must be fourth order on a smooth problem."""
    e1 = abs(integrate(rk4_step, 100.0) - ref)
    e2 = abs(integrate(rk4_step, 50.0) - ref)
    assert np.log2(e1 / e2) > 3.5, np.log2(e1 / e2)

for fn in (test_charge_conservation, test_thermal_equilibrium, test_second_law,
           test_ageing_is_monotone, test_determinism, test_integrator_order):
    fn(); print(f"  PASS  {fn.__name__:28s} {fn.__doc__.strip().splitlines()[0]}")
print("\n6 passed")

**Explore.** Move the controls and watch the result update. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as W
def explore(method="euler", dt=300, fan=0.0):
    step = {"euler": euler_step, "heun": heun_step, "rk4": rk4_step}[method]
    ts = np.arange(0, 7200 + dt, dt); x = np.array([36.0]); xs = [36.0]
    for t_now in ts[:-1]:
        x = step(thermal_only, t_now, x, dt, 40.0, fan, 35.0, PackParams()); xs.append(float(x[0]))
    ref = [integrate(rk4_step, 1.0, T_end=float(t_now), u=fan) if t_now > 0 else 36.0 for t_now in ts]
    plt.figure(figsize=(8, 3)); plt.plot(ts / 60, ref, label="reference"); plt.plot(ts / 60, np.clip(xs, 20, 80), "o-", ms=3, label=method)
    plt.xlabel("time (min)"); plt.ylabel("°C"); plt.legend(); plt.show()
    print(f"final error {abs(xs[-1] - ref[-1]):.2e} °C with {len(ts) - 1} steps; Euler limit {stability_limit(fan):.0f} s")
W.interact(explore, method=W.Dropdown(options=["euler", "heun", "rk4"], value="euler"),
           dt=W.IntSlider(value=300, min=10, max=2400, step=10), fan=W.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.1));

## Application challenges

Each challenge transfers the ideas of the day to a field of study. Choose the one closest to your programme and solve it in a copy of this notebook.

| Field | Challenge |
|---|---|
| Computer Science and Engineering | Implement the implicit Euler method for the thermal model, solving the update equation in closed form, and show that it stays stable for any step size. |
| Electronics and Communication Engineering | Add a latency of two samples to the sensor model, so that every reading arrives two readings late, and measure its effect on the scenario table of Day 1. |
| Electrical and Electronics Engineering | Simulate an RC circuit, a capacitor $C$ that charges through a resistor $R$, with the three integrators and measure their orders, comparing with the exact solution. The voltage of the capacitor follows the first-order equation of the lecture with the time constant $\tau = RC$. |
| Mechanical Engineering | Write the model of a damped spring-mass system, find its stability limit for explicit Euler and verify it numerically. The model has two quantities, the position and the velocity, and therefore two rate constants $\lambda$, its eigenvalues, which are complex numbers when the system oscillates. Explicit Euler is stable when the absolute value of $1 + \lambda\,\Delta t$ is at most 1 for both of them. |
| Biomedical Engineering | Simulate a two-compartment drug model, in which a dose passes from the gut into the blood at one rate constant and leaves the blood at another. Compare the time scales of absorption and elimination; is it stiff? |
| Civil Engineering | Turn the charging depot into a toll plaza with a varying arrival rate over the day and find the number of booths that keeps the mean wait under five minutes. |
| Aeronautical Engineering | Add one more physical test to section 8 that would catch a sign error in the ohmic heat term $I^2 R$, and show that it fails when the sign is flipped. |
| Biotechnology | Model the growth of a cell culture with the logistic equation $dN/dt = r N (1 - N/K)$, in which $N$ is the number of cells, $r$ the growth rate and $K$ the largest number that the culture can hold. Compare a fixed step with `solve_ivp` near the inflection point, where the growth is fastest. |

## After the notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-02/lab.html), tabs *Check yourself* and *Reflect and export*. The daily task and the research assignment are on the [day overview](https://github.com/utkukose/digital-twin-veltech-NB-lecture/tree/main/days/day-02).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

</div>